In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from collections import defaultdict
import os

# Resolve paths relative to this notebook's directory
_NOTEBOOK_DIR = os.path.dirname(os.path.abspath('mooncake-plot.ipynb'))

def load_mooncake_data(input_file):
    # If the path is already absolute, use it as-is; otherwise resolve relative to notebook dir
    if os.path.isabs(input_file):
        filepath = input_file
    else:
        filepath = os.path.join(_NOTEBOOK_DIR, input_file)
    data = []
    with open(filepath, 'r') as f:
        for line in f:
            data.append(json.loads(line.strip()))

    # Convert to DataFrame for easier analysis
    df = pd.DataFrame(data)

    # Convert timestamp from milliseconds to seconds
    df['timestamp_seconds'] = df['timestamp'] / 1000

    print(f"Dataset info:")
    print(f"Total requests: {len(df)}")

    # Create distribution plots
    fig, axes = plt.subplots(2, 2, figsize=(10, 6))
    workload_name = os.path.basename(input_file).split('.')[0]
    fig.suptitle(f'{workload_name} - Distribution Analysis', fontsize=16)

    # 1. RPS calculation and distribution
    # Group by second and count requests
    rps_data = df.groupby(df['timestamp_seconds'].astype(int)).size()
    print(f"RPS stats: min={rps_data.min()}, max={rps_data.max()}, mean={rps_data.mean():.2f}, std={rps_data.std():.2f}")

    # RPS distribution
    axes[0, 0].hist(rps_data.values, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0, 0].set_xlabel('Requests per Second')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('RPS Distribution')
    axes[0, 0].grid(True, alpha=0.3)

    # 2. Input token length distribution
    axes[0, 1].hist(df['input_length'], bins=50, alpha=0.7, color='lightgreen', edgecolor='black')
    axes[0, 1].set_xlabel('Input Token Length')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Input Token Length Distribution')
    axes[0, 1].grid(True, alpha=0.3)

    # 3. Output token length distribution
    axes[1, 0].hist(df['output_length'], bins=50, alpha=0.7, color='lightcoral', edgecolor='black')
    axes[1, 0].set_xlabel('Output Token Length')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Output Token Length Distribution')
    axes[1, 0].grid(True, alpha=0.3)

    # 4. Scatter plot of input vs output length
    scatter = axes[1, 1].scatter(df['input_length'], df['output_length'], alpha=0.5, s=20, c='purple')
    axes[1, 1].set_xlabel('Input Token Length')
    axes[1, 1].set_ylabel('Output Token Length')
    axes[1, 1].set_title('Input vs Output Token Length')
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    

    ####################################################################################################################################################################################
    ####################################################################################################################################################################################
    ####################################################################################################################################################################################
    
    # Create time series plots (1 second granularity) and token block distribution
    fig, axes = plt.subplots(3, 2, figsize=(10, 6))
    fig.suptitle(f'{workload_name} - Time Series Analysis (1 second granularity)', fontsize=16)

    # Calculate number of token blocks per request (each token block = 500 tokens)
    NUM_TOKEN_PER_HASH_ID = 100
    # NUM_TOKEN_PER_HASH_ID = 500 # og
    df['num_token_blocks'] = df['hash_ids'].apply(len)
    df['input_length_from_blocks'] = df['num_token_blocks'] * NUM_TOKEN_PER_HASH_ID

    # Prepare time series data
    max_time = int(df['timestamp_seconds'].max()) + 1
    time_bins = range(0, max_time + 1)

    # 1. RPS over time
    rps_by_second = df.groupby(df['timestamp_seconds'].astype(int)).size().reindex(time_bins, fill_value=0)

    axes[0, 0].plot(rps_by_second.index, rps_by_second.values, linewidth=1, alpha=0.8, color='blue')
    axes[0, 0].set_xlabel('Time (seconds)')
    axes[0, 0].set_ylabel('Requests per Second')
    axes[0, 0].set_title('RPS Over Time')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].set_xlim(0, max_time)

    # 2. Average input token length over time
    input_tokens_by_second = df.groupby(df['timestamp_seconds'].astype(int))['input_length'].mean().reindex(time_bins, fill_value=np.nan)

    # Only plot points where we have data (non-NaN values)
    valid_input_mask = ~input_tokens_by_second.isna()
    axes[0, 1].plot(input_tokens_by_second.index[valid_input_mask], input_tokens_by_second.values[valid_input_mask], linewidth=1, alpha=0.8, color='green')
    axes[0, 1].set_xlabel('Time (seconds)')
    axes[0, 1].set_ylabel('Average Input Token Length')
    axes[0, 1].set_title('Average Input Token Length Over Time')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].set_xlim(0, max_time)

    # 3. Average output token length over time
    output_tokens_by_second = df.groupby(df['timestamp_seconds'].astype(int))['output_length'].mean().reindex(time_bins, fill_value=np.nan)

    # Only plot points where we have data (non-NaN values)
    valid_output_mask = ~output_tokens_by_second.isna()
    axes[1, 0].plot(output_tokens_by_second.index[valid_output_mask], output_tokens_by_second.values[valid_output_mask], linewidth=1, alpha=0.8, color='red')
    axes[1, 0].set_xlabel('Time (seconds)')
    axes[1, 0].set_ylabel('Average Output Token Length')
    axes[1, 0].set_title('Average Output Token Length Over Time')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].set_xlim(0, max_time)

    # 4. Prefix sharing ratio over time (1-minute windows with radix tree approach)
    def calculate_prefix_sharing_ratio(requests_with_timestamps):
        if len(requests_with_timestamps) <= 1:
            return 0.0
        
        def find_longest_common_prefix(seq1, seq2):
            min_len = min(len(seq1), len(seq2))
            for i in range(min_len):
                if seq1[i] != seq2[i]:
                    return i
            return min_len
        
        sharing_ratios = []
        
        for i, (current_timestamp, current_hash_ids) in enumerate(requests_with_timestamps):
            if len(current_hash_ids) == 0:
                continue
            
            max_prefix_length = 0
            
            for j in range(i):
                prev_timestamp, prev_hash_ids = requests_with_timestamps[j]
                prefix_length = find_longest_common_prefix(current_hash_ids, prev_hash_ids)
                max_prefix_length = max(max_prefix_length, prefix_length)
            
            sharing_ratio = max_prefix_length / len(current_hash_ids) if len(current_hash_ids) > 0 else 0.0
            sharing_ratios.append(sharing_ratio)
        
        return sum(sharing_ratios) / len(sharing_ratios) if sharing_ratios else 0.0

    # Calculate prefix sharing ratio per minute
    max_minute = int(df['timestamp_seconds'].max() / 60) + 1
    minute_bins = range(0, max_minute + 1)

    prefix_ratios_by_minute = []
    minute_timestamps = []

    for minute in minute_bins:
        minute_data = df[(df['timestamp_seconds'] >= minute * 60) & (df['timestamp_seconds'] < (minute + 1) * 60)]
        if len(minute_data) > 1:
            minute_data_sorted = minute_data.sort_values('timestamp_seconds')
            requests_with_timestamps = list(zip(minute_data_sorted['timestamp_seconds'], minute_data_sorted['hash_ids']))
            ratio = calculate_prefix_sharing_ratio(requests_with_timestamps)
            prefix_ratios_by_minute.append(ratio)
            minute_timestamps.append(minute)

    axes[1, 1].plot(minute_timestamps, prefix_ratios_by_minute, linewidth=2, alpha=0.8, color='purple', marker='o', markersize=4)
    axes[1, 1].set_xlabel('Time (minutes)')
    axes[1, 1].set_ylabel('Prefix Sharing Ratio')
    axes[1, 1].set_title('Prefix Cache Hit Ratio Over Time (1-minute windows)')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].set_xlim(0, max_minute)
    axes[1, 1].set_ylim(0, 1)

    # 5. Token blocks per request distribution
    axes[2, 0].hist(df['num_token_blocks'], bins=30, alpha=0.7, color='orange', edgecolor='black')
    axes[2, 0].set_xlabel('Number of Token Blocks')
    axes[2, 0].set_ylabel('Frequency')
    axes[2, 0].set_title('Token Blocks per Request Distribution')
    axes[2, 0].grid(True, alpha=0.3)

    # Hide the empty subplot
    axes[2, 1].set_visible(False)

    plt.tight_layout()
    plt.show()


load_mooncake_data('Mooncake_conversation_trace.jsonl')
load_mooncake_data('Mooncake_toolagent_trace.jsonl')
load_mooncake_data('Mooncake_synthetic_trace.jsonl')

In [ ]:
load_mooncake_data('Mooncake_conversation_trace.jsonl')
load_mooncake_data('Mooncake_toolagent_trace.jsonl')
load_mooncake_data('Mooncake_synthetic_trace.jsonl')